# Chatbot And RAG Evaluation
Retrieval Augmented Generation (RAG) is a technique that enhances Large Language Models (LLMs) by providing them with relevant external knowledge. It has become one of the most widely used approaches for building LLM applications.

This tutorial will show you how to evaluate your RAG applications using LangSmith. You'll learn:

- How to create test datasets
- How to run your RAG application on those datasets
- How to measure your application's performance using different evaluation metrics
## Overview
A typical RAG evaluation workflow consists of three main steps:

- Creating a dataset with questions and their expected answers
- Running your RAG application on those questions
- Using evaluators to measure how well your application performed, looking at factors like:

1. Answer relevance
2. Answer accuracy
3. Retrieval quality

For this tutorial, we'll create and evaluate a bot that answers questions about a few of Lilian Weng's insightful blog posts.

# Chatbot Evaluation

In [4]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["LANGSMITH_API_KEY"]=os.getenv("LANGSMITH_API_KEY")

In [8]:
from langsmith import Client

client = Client()

# Define dataset: these are your test cases
dataset_name = "Chatbots Evaluation"
dataset = client.read_dataset(dataset_name=dataset_name)
client.create_examples(
    dataset_id=dataset.id,
    examples=[
        {
            "inputs": {"question": "What is LangChain?"},
            "outputs": {"answer": "A framework for building LLM applications"},
        },
        {
            "inputs": {"question": "What is LangSmith?"},
            "outputs": {"answer": "A platform for observing and evaluating LLM applications"},
        },
        {
            "inputs": {"question": "What is OpenAI?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        },
        {
            "inputs": {"question": "What is Google?"},
            "outputs": {"answer": "A technology company known for search"},
        },
        {
            "inputs": {"question": "What is Mistral?"},
            "outputs": {"answer": "A company that creates Large Language Models"},
        }
    ]
)

{'example_ids': ['4f2be238-391f-4b5e-850f-d0d6bbeb69b3',
  'ed4f87d4-2a3c-4a01-99c1-7653774ad27d',
  'e1868800-62e6-4a80-af2b-b3afc63faf7d',
  '89c9b72d-1e3b-4dd9-a26c-65c27958ead5',
  '77fb85a6-3e06-4e51-ba73-16d237f00089'],
 'count': 5,
 'as_of': '2026-06-06T04:46:53.404355179Z'}

## Define Metrics (LLM As A Judge)

In [15]:
from langsmith import traceable
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

eval_instructions = (
    "You are an expert professor specialized in grading students' answers to questions."
)

@traceable(name="correctness_evaluator")
def correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
):
    prompt = f"""
You are grading the following question:
{inputs['question']}

Here is the reference answer:
{reference_outputs['answer']}

Here is the predicted answer:
{outputs['response']}

Respond with only:
CORRECT
or
INCORRECT
"""

    response = llm.invoke(
        [
            ("system", eval_instructions),
            ("human", prompt),
        ]
    )

    grade = response.content.strip().upper()

    return {
        "score": 1 if grade == "CORRECT" else 0,
        "key": "correctness",
    }

In [17]:
## Concisions- checks wheather the actual output is less than 2x the length of the expected output

def concision(outputs, reference_outputs):
    response_text = getattr(outputs, "response", outputs.get("response"))
    reference_text = getattr(reference_outputs, "answer", reference_outputs.get("answer"))

    return {
        "key": "concision",
        "score": 1 if len(response_text) <= 2 * len(reference_text) else 0
    }

## Run Evaluations

In [18]:
default_instructions = "Respond to the users question in a short, concise manner (one short sentence)."
def my_app(question: str, model=llm, instructions: str = default_instructions) -> str:
    response = model.invoke(
        [
            ("system", instructions),
            ("human", question),
        ]
    )

    return response.content.strip()

In [19]:
### Call my_app for every datapoints
def ls_target(inputs: str) -> dict:
    return {"response": my_app(inputs["question"])}

In [20]:
## Run our evaluation
experiment_results=client.evaluate(
    ls_target, ## Your AI system
    data=dataset_name,
    evaluators=[correctness,concision],
    experiment_prefix="gemini-chatbot"
)

View the evaluation results for experiment: 'gemini-chatbot-735cec42' at:
https://smith.langchain.com/o/26d94213-aa7d-4d7c-8cdf-497fccf6ec4f/datasets/3e46145e-dcb3-43be-bf15-2b5d83d16165/compare?selectedSessions=c41f651a-ff27-434d-89a9-1b7fba0668c1




1it [00:08,  8.12s/it]Error running evaluator <DynamicRunEvaluator correctness> on run 019e9b5b-7ae7-7453-b055-67525ba984da: ChatGoogleGenerativeAIError("Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\\nPlease retry in 40.65462179s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'gene